# OpenAI API 연구 노트북

`.env`의 `OPENAI_API_KEY`를 로드해서 ChatGPT API를 테스트하는 스타터 노트북입니다.

In [1]:
import os

from dotenv import load_dotenv
from openai import OpenAI

# 노트북은 notebooks/ 안에 있으므로 프로젝트 루트의 .env를 로드
load_dotenv("../.env")

# 모든 실험에서 공통으로 쓸 기본 모델
MODEL = "gpt-5-nano"

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("API 키 로드됨:", bool(os.getenv("OPENAI_API_KEY")))

API 키 로드됨: True


## 연결 테스트

In [2]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "안녕! 연결 테스트야. 한 문장으로 대답해줘."}],
)
print(response.choices[0].message.content)

안녕, 연결이 정상적으로 확인되었습니다.


## 재사용 헬퍼

실험할 때 간단히 부를 수 있는 함수입니다.

In [3]:
def ask(prompt: str, model: str = MODEL, system: str | None = None) -> str:
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content


print(ask("파이썬으로 피보나치 수열을 만드는 가장 짧은 방법은?"))

가장 짧은 방법은 용도에 따라 다르지만, 일반적으로 많이 쓰는 간결한 버전은 아래처럼 출력용 루프나 제너레이터를 쓰는 방식입니다.

1) 처음 n개를 출력하는 가장 짧은 형태(읽기보단 간결):
n = int(input())
a, b = 0, 1
for _ in range(n):
    print(a, end=' ')
    a, b = b, a + b
print()

2) 피보나치 수열의 값들을 리스트로 얻고 싶을 때:
def fib(n):
    a, b = 0, 1
    res = []
    for _ in range(n):
        res.append(a)
        a, b = b, a + b
    return res

3) 피보나치 수열을 제너레이터로 만들 때(필요 시 하나씩 얻기 좋음):
def fib(n):
    a, b = 0, 1
    for _ in range(n):
        yield a
        a, b = b, a + b

참고
- 가장 짧은 코드라도 읽기 쉬운 버전이 필요하면, 제너레이터 버전이 보통 가장 균형 잡힌 선택입니다.
- 재귀로 구현하면 코드가 더 짧아 보일 수 있지만, 시간 복잡도가 급격히 늘어나고 실용적이지 않을 수 있습니다. 가능한 한 DP/제너레이터 방식이 선호됩니다.


## 도구 호출(Function Calling) — 에이전트 루프

모델이 직접 답하지 않고, 우리가 정의한 **도구(함수)**를 호출해서 정보를 얻은 뒤 그 결과를 보고 다음 행동을 정하는 방식이다.

흐름은 이렇다.
1. 모델에게 질문과 함께 **도구 명세**(어떤 함수가 있고 인자가 무엇인지)를 넘긴다.
2. 모델이 답 대신 **"이 함수를 이런 인자로 불러라"(tool_calls)** 를 반환하면, 우리가 실제 함수를 실행한다.
3. 함수 **결과를 다시 모델에게 돌려준다**. 모델은 그 결과를 보고 또 도구를 부를지, 최종 답을 낼지 정한다.
4. 모델이 더 이상 도구를 부르지 않고 최종 답을 내면 반복을 멈춘다.

아래 예시는 **"서울과 부산 중 인구가 더 많은 도시의 현재 기온"** 을 묻는다.
마지막 기온 조회는 앞의 인구 비교 결과에 따라 대상 도시가 정해지므로, **도구 결과를 보고 다음 행동을 결정**하는 과정을 그대로 보여준다. (총 3번 호출: 인구 2번 → 기온 1번)

In [4]:
# 1) 도구 명세 — 모델에게 "이런 함수를 쓸 수 있다"고 알려주는 부분
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_population",
            "description": "주어진 도시의 인구수를 반환한다.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string", "description": "도시 이름"}},
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "주어진 도시의 현재 기온(섭씨)을 반환한다.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string", "description": "도시 이름"}},
                "required": ["city"],
            },
        },
    },
]

# 2) 실제 함수 구현 — 재현 가능하도록 가짜 데이터 사용
POPULATION = {"서울": 9_600_000, "부산": 3_300_000, "대구": 2_400_000}
WEATHER = {"서울": 29, "부산": 26, "대구": 31}


def _norm(city):
    # 모델이 도시명을 영어/대소문자로 넘겨도 인식하도록 정규화
    return {"seoul": "서울", "busan": "부산", "daegu": "대구"}.get(city.strip().lower(), city.strip())


def get_population(city):
    c = _norm(city)
    return {"city": c, "population": POPULATION.get(c)}


def get_weather(city):
    c = _norm(city)
    return {"city": c, "temp_c": WEATHER.get(c)}


# 함수 이름 -> 실제 파이썬 함수 매핑
IMPL = {"get_population": get_population, "get_weather": get_weather}

In [5]:
import json

# 도구를 추측 없이 반드시 호출하게 만드는 시스템 지시
TOOL_SYSTEM = (
    "너는 도구를 사용하는 도우미다. 인구·기온 같은 사실 정보는 절대 추측하지 말고, "
    "반드시 제공된 도구(get_population, get_weather)를 호출해서 확인해라. "
    "필요한 정보를 모두 도구로 확인한 뒤에만 최종 답을 말해라."
)


def run_agent(user_message, max_steps=6):
    """모델 호출 -> 도구 실행 -> 결과 전달을 반복하는 에이전트 루프."""
    messages = [
        {"role": "system", "content": TOOL_SYSTEM},
        {"role": "user", "content": user_message},
    ]
    calls = 0
    for step in range(1, max_steps + 1):
        resp = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        msg = resp.choices[0].message

        # 도구 호출이 없으면 = 최종 답변
        if not msg.tool_calls:
            print(f"[{step}단계] 최종 답변:\n{msg.content}")
            return msg.content

        # 도구 호출이 있으면: 모델의 요청(assistant turn)을 기록하고 각 도구를 실행
        messages.append(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            result = IMPL[tc.function.name](**args)
            calls += 1
            print(f"[{step}단계] 도구호출 #{calls}: {tc.function.name}({args}) -> {result}")
            # 실행 결과를 role=tool 메시지로 모델에게 돌려줌
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result, ensure_ascii=False),
            })
    print("최대 단계에 도달해 중단")

In [6]:
# 인구를 먼저 비교(2번) -> 더 많은 쪽(서울)의 기온 조회(1번) -> 최종 답변
run_agent("서울과 부산 중에서 인구가 더 많은 도시의 현재 기온을 알려줘.")

[1단계] 도구호출 #1: get_population({'city': '서울'}) -> {'city': '서울', 'population': 9600000}
[1단계] 도구호출 #2: get_population({'city': '부산'}) -> {'city': '부산', 'population': 3300000}
[2단계] 도구호출 #3: get_weather({'city': '서울'}) -> {'city': '서울', 'temp_c': 29}
[3단계] 최종 답변:
더 많은 인구를 가진 도시는 서울이고, 현재 기온은 29°C입니다.


'더 많은 인구를 가진 도시는 서울이고, 현재 기온은 29°C입니다.'

**관찰 포인트**
- 각 `[N단계]` 줄에서 모델이 어떤 도구를 어떤 인자로 불렀는지, 그 결과가 무엇인지 볼 수 있다.
- 마지막 `get_weather` 호출 대상이 앞의 인구 조회 결과에 따라 정해진다는 점이 핵심이다 — **결과를 보고 다음 행동을 정한다**.
- 질문을 "세 도시 중 가장 더운 곳" 처럼 바꾸면 호출 횟수와 순서가 어떻게 달라지는지도 실험해 볼 수 있다.

## 도구 스키마 자동 생성 — 파이썬 함수에서 (direct tool calling)

> 정정: 이 섹션은 OpenAI가 말하는 **"Programmatic Tool Calling"이 아니다.** 그건 완전히 다른 별개 기능이고, 이 노트북 맨 아래에서 따로 설명한다.
> 여기서 하는 건 앞의 에이전트 루프와 **똑같은 일반(direct) 함수 호출**인데, 도구 스키마(JSON)를 손으로 안 쓰고 자동으로 만들어 쓰는 편의 기법일 뿐이다.

위 예시는 도구 명세(JSON 스키마)를 손으로 직접 작성했다. 도구가 늘어나면 스키마와 실제 함수를 따로 관리해야 해서 번거롭고 실수가 나기 쉽다.

여기서는 **파이썬 함수 하나만 쓰면 도구 스키마가 자동으로 만들어지는** 방식을 쓴다.
`@tool` 데코레이터가 함수의 **타입 힌트**와 **docstring**을 읽어 OpenAI 도구 명세를 자동 생성하고, 함수 이름으로 실제 실행까지 연결한다. 새 도구를 추가하려면 함수 위에 `@tool` 만 붙이면 된다.

In [7]:
import inspect
import json
from typing import get_type_hints

# 파이썬 타입 -> JSON 스키마 타입
_PYTYPE = {str: "string", int: "integer", float: "number", bool: "boolean"}
TOOLBOX = {}  # 함수이름 -> (실제 함수, 자동 생성된 스키마)


def tool(func):
    """함수의 타입 힌트 + docstring에서 OpenAI 도구 스키마를 자동 생성해 등록하는 데코레이터."""
    hints = get_type_hints(func)
    props, required = {}, []
    for name, param in inspect.signature(func).parameters.items():
        props[name] = {"type": _PYTYPE.get(hints.get(name, str), "string")}
        if param.default is inspect.Parameter.empty:  # 기본값이 없으면 필수 인자
            required.append(name)
    schema = {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": (func.__doc__ or "").strip(),
            "parameters": {"type": "object", "properties": props, "required": required},
        },
    }
    TOOLBOX[func.__name__] = (func, schema)
    return func


# --- 도구는 그냥 평범한 파이썬 함수 + @tool 만 붙이면 끝 ---
@tool
def convert_currency(amount: float, base: str, quote: str) -> float:
    """amount만큼의 base 통화를 quote 통화로 환산한 금액을 반환한다. (예: base='USD', quote='KRW')"""
    rate = {("USD", "KRW"): 1385.0, ("EUR", "KRW"): 1500.0}.get((base.upper(), quote.upper()))
    return None if rate is None else round(amount * rate, 2)


@tool
def add(a: float, b: float) -> float:
    """두 수를 더한 값을 반환한다."""
    return a + b


# 손으로 안 썼는데도 스키마가 만들어져 있다
print(json.dumps([s for _, s in TOOLBOX.values()], ensure_ascii=False, indent=2))

[
  {
    "type": "function",
    "function": {
      "name": "convert_currency",
      "description": "amount만큼의 base 통화를 quote 통화로 환산한 금액을 반환한다. (예: base='USD', quote='KRW')",
      "parameters": {
        "type": "object",
        "properties": {
          "amount": {
            "type": "number"
          },
          "base": {
            "type": "string"
          },
          "quote": {
            "type": "string"
          }
        },
        "required": [
          "amount",
          "base",
          "quote"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "add",
      "description": "두 수를 더한 값을 반환한다.",
      "parameters": {
        "type": "object",
        "properties": {
          "a": {
            "type": "number"
          },
          "b": {
            "type": "number"
          }
        },
        "required": [
          "a",
          "b"
        ]
      }
    }
  }
]


In [8]:
# 등록된 도구 전체를 모델에 넘기고, 반환된 tool_calls 를 이름으로 디스패치
prog_tools = [s for _, s in TOOLBOX.values()]
PROG_SYSTEM = "사실 계산은 추측하지 말고 반드시 제공된 도구를 호출해서 처리해라."


def run_tools(user_message, max_steps=6):
    messages = [
        {"role": "system", "content": PROG_SYSTEM},
        {"role": "user", "content": user_message},
    ]
    for step in range(1, max_steps + 1):
        msg = client.chat.completions.create(
            model=MODEL, messages=messages, tools=prog_tools
        ).choices[0].message
        if not msg.tool_calls:
            print(f"[{step}단계] 최종 답변: {msg.content}")
            return msg.content
        messages.append(msg)
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            result = TOOLBOX[tc.function.name][0](**args)  # 이름 -> 실제 함수 자동 연결
            print(f"[{step}단계] {tc.function.name}({args}) -> {result}")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result, ensure_ascii=False),
            })
    print("최대 단계에 도달해 중단")


# convert_currency 로 100달러를 원화로 -> 그 결과에 add 로 수수료 5000원 더하기
run_tools("100 미국 달러를 원화로 바꾸고, 거기에 수수료 5000원을 더하면 총 얼마야?")

[1단계] convert_currency({'amount': 100, 'base': 'USD', 'quote': 'KRW'}) -> 138500.0
[2단계] add({'a': 138500, 'b': 5000}) -> 143500
[3단계] 최종 답변: 143,500 KRW

구현 결과:
- 100 USD를 KRW로 환산: 138,500 KRW
- 여기에 수수료 5,000 KRW 추가: 143,500 KRW

필요한 다른 환전 계산이 있나요?


'143,500 KRW\n\n구현 결과:\n- 100 USD를 KRW로 환산: 138,500 KRW\n- 여기에 수수료 5,000 KRW 추가: 143,500 KRW\n\n필요한 다른 환전 계산이 있나요?'

## Programmatic Tool Calling — 실제 기능 (gpt-5.6에서 동작 확인)

앞 "도구 스키마 자동 생성" 섹션은 이 기능이 **아니다**(그건 일반 direct 호출이다). OpenAI의 **Programmatic Tool Calling**은 별개 기능이다.

**무엇인가** — 모델이 도구를 하나씩 직접 부르는 대신, **모델이 JavaScript 코드를 생성**하고 그 코드가 OpenAI의 격리된 런타임(V8)에서 실행되면서 도구들을 조율한다. 코드가 루프·조건·병렬 호출을 처리하고, 중간 결과를 런타임에 담아두었다가 **작게 요약된 결과만** 돌려준다.

**Direct(일반) 호출과의 차이**

| | Direct (앞의 예시) | Programmatic |
|---|---|---|
| 도구를 부르는 주체 | 모델이 매 턴 직접 | 모델이 생성한 JS 코드 |
| 제어 흐름 | 결과마다 모델이 새로 판단 | 코드가 루프/조건으로 처리 |
| 중간 결과 | 전부 컨텍스트에 쌓임 | 코드가 필터/집계 후 요약만 반환 |
| 실행 위치 | 내 클라이언트 | OpenAI 호스팅 런타임 |

**ReAct 질문에 대한 답** — 오히려 반대다. "도구 결과를 보고 다음 행동을 모델이 새로 판단"하는 건 **direct 호출**(= ReAct 스타일, 앞의 예시들)이다. Programmatic은 그 판단을 **코드로 미리 정해두는** 방식이라, 결과를 필터/정렬/집계하는 예측 가능한 흐름에 강하고, 매 결과마다 모델의 새 판단이 필요한 일에는 오히려 안 맞는다.

**지원 모델 (직접 검증함)** — `gpt-5.6-sol` / `gpt-5.6-luna` / `gpt-5.6-terra` 는 **지원 O**. 반면 이 노트북 기본 모델 `gpt-5-nano` 는 물론 `gpt-5`, `gpt-4o`, `gpt-5.1`~`gpt-5.5`, `gpt-5-pro`, `gpt-5-codex` 는 전부 `Tool 'programmatic_tool_calling' is not supported` 400 에러가 난다. **즉 이 기능은 `gpt-5.6` 계열에서만 쓸 수 있다.**

**설정 (Responses API 전용)**
- `tools` 에 `{"type": "programmatic_tool_calling"}` 추가
- 각 함수 도구에 `allowed_callers`(`["programmatic"]` / `["direct"]` / 둘 다)와 `output_schema` 지정
- `client.chat.completions` 가 아니라 `client.responses.create(...)` 사용

아래 셀이 실제 동작 예시다. (재고 5개를 조회해 100개 이상만 필터링 — 모델이 그 JS를 직접 작성한다)

출처: OpenAI 공식 문서 — Programmatic Tool Calling (`developers.openai.com/api/docs/guides/tools-programmatic-tool-calling`)

In [9]:
# Programmatic Tool Calling 실제 예시 — Responses API + gpt-5.6 계열
# (gpt-5-nano 는 이 기능 미지원. 반드시 gpt-5.6 모델을 써야 한다)
import json

PTC_MODEL = "gpt-5.6-luna"  # gpt-5.6-luna / gpt-5.6-terra 도 가능

ptc_tools = [
    {
        "type": "function",
        "name": "get_inventory",
        "description": "주어진 SKU의 재고 수량을 반환한다.",
        "parameters": {"type": "object", "properties": {"sku": {"type": "string"}}, "required": ["sku"]},
        # output_schema: 생성된 JS 가 반환 필드를 신뢰하고 다룰 수 있게 명시
        "output_schema": {
            "type": "object",
            "properties": {"sku": {"type": "string"}, "available_units": {"type": "number"}},
        },
        "allowed_callers": ["programmatic"],  # 모델이 만든 프로그램(JS)에서만 호출 허용
    },
    {"type": "programmatic_tool_calling"},     # 호스팅 오케스트레이션 활성화
]

STOCK = {"sku_1": 120, "sku_2": 30, "sku_3": 200, "sku_4": 5, "sku_5": 88}

items = [{"role": "user", "content": "sku_1 부터 sku_5 까지 재고를 확인해서, 100개 이상인 SKU만 알려줘."}]

for step in range(1, 8):
    resp = client.responses.create(model=PTC_MODEL, store=False, input=items, tools=ptc_tools)
    items.extend(it.model_dump(exclude_none=True) for it in resp.output)

    # 모델이 생성한 JS 프로그램 확인 (한 번만 나온다)
    for it in resp.output:
        if it.type == "program" and getattr(it, "code", None):
            print("[모델이 생성한 프로그램]\n" + it.code + "\n")

    # 프로그램이 요청한 함수 호출을 우리가 실행해서 결과를 돌려줌 (caller.type == 'program')
    calls = [it for it in resp.output if it.type == "function_call"]
    if not calls:
        if any(it.type == "message" for it in resp.output):
            print("=== 최종 답변 ===\n" + resp.output_text)
            break
        continue
    for call in calls:
        args = json.loads(call.arguments)
        result = {"sku": args["sku"], "available_units": STOCK.get(args["sku"], 0)}
        caller = getattr(call, "caller", None)
        print(f"  program 호출: get_inventory({args}) -> {result}")
        items.append({
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": json.dumps(result),
            "caller": caller.model_dump() if caller else None,
        })

[모델이 생성한 프로그램]
const skus = ["sku_1","sku_2","sku_3","sku_4","sku_5"];
const results = [];
for (const sku of skus) {
  results.push(await tools.get_inventory({sku}));
}
text(JSON.stringify(results));


  program 호출: get_inventory({'sku': 'sku_1'}) -> {'sku': 'sku_1', 'available_units': 120}
  program 호출: get_inventory({'sku': 'sku_2'}) -> {'sku': 'sku_2', 'available_units': 30}
  program 호출: get_inventory({'sku': 'sku_3'}) -> {'sku': 'sku_3', 'available_units': 200}
  program 호출: get_inventory({'sku': 'sku_4'}) -> {'sku': 'sku_4', 'available_units': 5}
  program 호출: get_inventory({'sku': 'sku_5'}) -> {'sku': 'sku_5', 'available_units': 88}
=== 최종 답변 ===
재고가 100개 이상인 SKU는 다음과 같습니다.

- **sku_1:** 120개
- **sku_3:** 200개


## 여기서부터 자유롭게 연구/실험

아래 셀부터 원하는 실험을 진행하세요.